In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/feature_engineered_data.csv"
)

df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,invoiceno_length,stockcode_length,description_length,invoicedate_length,country_length
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,6,6,34,19,14
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,6,5,19,19,14
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,6,6,30,19,14
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,6,6,35,19,14
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,6,6,30,19,14


In [4]:
df.columns.tolist()

['invoiceno',
 'stockcode',
 'description',
 'quantity',
 'invoicedate',
 'unitprice',
 'customerid',
 'country',
 'invoiceno_length',
 'stockcode_length',
 'description_length',
 'invoicedate_length',
 'country_length']

In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [6]:
from src.insights.kpi_engine import (
    generate_basic_kpis,
    generate_numeric_kpis,
    get_top_categories,
    generate_insights
)

In [7]:
basic_kpis = generate_basic_kpis(df)

basic_kpis

{'total_rows': 536641,
 'total_columns': 13,
 'duplicate_rows': np.int64(0),
 'total_missing_values': np.int64(0),
 'numeric_columns': 8,
 'categorical_columns': 5}

In [8]:
numeric_columns = df.select_dtypes(
    include="number"
).columns

numeric_kpi_table = pd.DataFrame({
    "column": numeric_columns,
    "sum": [df[col].sum() for col in numeric_columns],
    "mean": [df[col].mean() for col in numeric_columns],
    "median": [df[col].median() for col in numeric_columns],
    "minimum": [df[col].min() for col in numeric_columns],
    "maximum": [df[col].max() for col in numeric_columns]
})

numeric_kpi_table

,column,sum,mean,median,minimum,maximum
0,quantity,5.162502e+06,9.620029,3.00,-80995.00,80995.0
1,unitprice,2.486073e+06,4.632656,2.08,-11062.06,38970.0
2,customerid,8.182111e+09,15246.898157,15145.00,12346.00,18287.0
3,invoiceno_length,3.229100e+06,6.017244,6.00,6.00,7.0
4,stockcode_length,2.729911e+06,5.087034,5.00,1.00,12.0
5,description_length,1.419033e+07,26.442866,27.00,1.00,35.0
6,invoicedate_length,1.019618e+07,19.000000,19.00,19.00,19.0
7,country_length,7.175565e+06,13.371258,14.00,3.00,20.0


In [9]:
from src.analytics.pandas_analysis import get_top_values

In [10]:
top_countries = get_top_values(
    df,
    "country",
    10
)

top_countries

country
United Kingdom    490300
Germany             9480
France              8541
EIRE                8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: count, dtype: int64

In [12]:
total_quantity = df["quantity"].sum()

total_quantity

np.int64(5162502)

In [14]:
country_quantity = (
    df.groupby("country")["quantity"]
    .sum()
    .reset_index()
)

In [16]:
country_quantity["contribution_percent"] = (
    country_quantity["quantity"]
    / country_quantity["quantity"].sum()
    * 100
)

In [17]:
country_quantity.head()

,country,quantity,contribution_percent
0,Australia,83643,1.620203
1,Austria,4827,0.093501
2,Bahrain,260,0.005036
3,Belgium,23152,0.448465
4,Brazil,356,0.006896


In [19]:
country_quantity["rank"] = (
    country_quantity["quantity"]
    .rank(
        ascending=False,
        method="dense"
    )
)

country_quantity = country_quantity.sort_values(
    "rank"
)

country_quantity.head(10)

,country,quantity,contribution_percent,rank
36,United Kingdom,4250328,82.330777,1.0
24,Netherlands,200128,3.876570,2.0
10,EIRE,142495,2.760193,3.0
14,Germany,117341,2.272948,4.0
13,France,110438,2.139234,5.0
0,Australia,83643,1.620203,6.0
32,Sweden,35632,0.690208,7.0
33,Switzerland,30313,0.587177,8.0
31,Spain,26817,0.519457,9.0
20,Japan,25218,0.488484,10.0


In [20]:
top_10 = country_quantity.head(10)

top_10

,country,quantity,contribution_percent,rank
36,United Kingdom,4250328,82.330777,1.0
24,Netherlands,200128,3.876570,2.0
10,EIRE,142495,2.760193,3.0
14,Germany,117341,2.272948,4.0
13,France,110438,2.139234,5.0
0,Australia,83643,1.620203,6.0
32,Sweden,35632,0.690208,7.0
33,Switzerland,30313,0.587177,8.0
31,Spain,26817,0.519457,9.0
20,Japan,25218,0.488484,10.0


In [21]:
top_10 = country_quantity.tail(10)

top_10

,country,quantity,contribution_percent,rank
35,United Arab Emirates,982,0.019022,29.0
23,Malta,944,0.018286,30.0
22,Lithuania,652,0.012630,31.0
8,Czech Republic,592,0.011467,32.0
11,European Community,497,0.009627,33.0
21,Lebanon,386,0.007477,34.0
4,Brazil,356,0.006896,35.0
28,RSA,352,0.006818,36.0
2,Bahrain,260,0.005036,37.0
29,Saudi Arabia,75,0.001453,38.0


In [8]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [9]:
from src.insights.kpi_engine import (
    generate_basic_kpis,
    generate_numeric_kpis,
    get_top_categories,
    generate_insights,
    calculate_contribution,
    rank_values,
    get_top_n,
    get_bottom_n
)

In [10]:
test_df = pd.DataFrame({
    "country": ["A", "B", "C", "D"],
    "sales": [500, 300, 150, 50]
})

In [12]:
test_df["contribution"] = calculate_contribution(
    test_df,
    "sales"
)

test_df

,country,sales,contribution
0,A,500,50.0
1,B,300,30.0
2,C,150,15.0
3,D,50,5.0


In [14]:
test_df["rank"] = rank_values(
    test_df,
    "sales"
)

test_df

,country,sales,contribution,rank
0,A,500,50.0,1.0
1,B,300,30.0,2.0
2,C,150,15.0,3.0
3,D,50,5.0,4.0


In [16]:
get_top_n(
    test_df,
    "sales",
    2
)

,country,sales,contribution,rank
0,A,500,50.0,1.0
1,B,300,30.0,2.0


In [17]:
get_bottom_n(
    test_df,
    "sales",
    2
)

,country,sales,contribution,rank
3,D,50,5.0,4.0
2,C,150,15.0,3.0


In [18]:
kpi_report = pd.DataFrame({
    "KPI": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Missing Values",
        "Numeric Columns",
        "Categorical Columns"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isnull().sum().sum(),
        len(df.select_dtypes(include="number").columns),
        len(df.select_dtypes(
            include=["object", "string", "category"]
        ).columns)
    ]
})

kpi_report

,KPI,Value
0,Total Rows,536641
1,Total Columns,13
2,Duplicate Rows,0
3,Missing Values,0
4,Numeric Columns,8
5,Categorical Columns,5


In [19]:
kpi_report = pd.DataFrame({
    "KPI": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Missing Values",
        "Numeric Columns",
        "Categorical Columns"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isnull().sum().sum(),
        len(df.select_dtypes(include="number").columns),
        len(df.select_dtypes(
            include=["object", "string", "category"]
        ).columns)
    ]
})

kpi_report

,KPI,Value
0,Total Rows,536641
1,Total Columns,13
2,Duplicate Rows,0
3,Missing Values,0
4,Numeric Columns,8
5,Categorical Columns,5


In [20]:
kpi_report.to_csv(
    "../data/processed/kpi_report_v2.csv",
    index=False
)

In [21]:
check = pd.read_csv(
    "../data/processed/kpi_report_v2.csv"
)

check

,KPI,Value
0,Total Rows,536641
1,Total Columns,13
2,Duplicate Rows,0
3,Missing Values,0
4,Numeric Columns,8
5,Categorical Columns,5
